# 02 - Nominal 24-h operating trajectory and network operation without storage (B0)

**Question:** what is the common nominal trajectory (Case-3 demand, DC-side PV and wind, tariff)
supplied to every strategy, and how does the validated network operate with the new renewables
but without a BESS?

* Demand: Load Case 3 (original tool, hourly factors interpolated, interval midpoints).
* PV (new): clear-sky (Haurwitz) output, Seville, 15 April, horizontal modules, PR = 0.85,
  distributed over the 10 DC load buses in proportion to their Case-3 DC rating.
* Wind (new): diurnal wind-speed profile through a cubic power curve, at DC bus D19.
* Network-security layer: minimum uniform renewable curtailment that removes
  renewable-driven violations (common to all strategies).

Outputs: `output/results/results_profiles.csv`, `output/results/results_B0_renewables.csv`,
`output/experiment_figures/Fig_operating_profiles.pdf`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dfm_acdc.config import RESULTS_DIR, FIG_DIR, RenewableParams, EMSParams, BESSParams
from dfm_acdc.network import Topology3
from dfm_acdc.oracle import NetworkOracle
from dfm_acdc import profiles, plotstyle

plotstyle.apply_style()
net = Topology3()
ren, ems = RenewableParams(), EMSParams()
day = profiles.build_day(net, ren, ems)
dt = ems.dt_h

In [ ]:
prof = pd.DataFrame({"t_h": day.t0, "t_mid_h": day.tm, "P_ac_kW": day.P_ac, "Q_ac_kvar": day.Q_ac,
                     "P_dc_kW": day.P_dc, "P_pv_kW": day.P_pv, "P_wt_kW": day.P_wt, "price_EUR_kWh": day.price})
prof.to_csv(os.path.join(RESULTS_DIR, "results_profiles.csv"), index=False)
E = lambda x: float(np.sum(x) * dt)
print(f"Daily energy: AC demand {E(day.P_ac):.0f} kWh, DC demand {E(day.P_dc):.0f} kWh "
      f"(DC share {E(day.P_dc)/E(day.P_load)*100:.1f} %), PV {E(day.P_pv):.0f} kWh, wind {E(day.P_wt):.0f} kWh")
print(f"Peaks: total demand {day.P_load.max():.0f} kW at {day.t0[day.P_load.argmax()]:.2f} h, "
      f"DC demand {day.P_dc.max():.0f} kW, PV {day.P_pv.max():.0f} kW, wind {day.P_wt.max():.1f} kW")
print("PV allocation (kWp):", {b: round(s * ren.pv_rated_kwp, 1) for b, s in day.pv_shares.items()})

## B0 -- network with renewables, no storage

In [ ]:
orc = NetworkOracle(net, day, ren.wt_bus, BESSParams().P_rated_kw)
rows = []
for k in range(ems.n_steps_day):
    r0 = orc.evaluate(k, 0.0, 0.0)
    a, ra = orc.min_curtailment(k, 0.0)
    rows.append({"t_h": day.t0[k], "e_max_nocurt": r0["e"].max(), "bind_nocurt": orc.binding_group(r0["e"]),
                 "alpha": a, "P_curt_kW": ra["P_curt"], "e_max": ra["e"].max(), "P_MV0_kW": ra["P_MV0"],
                 "loss_kW": ra["loss"]})
b0 = pd.DataFrame(rows)
b0.to_csv(os.path.join(RESULTS_DIR, "results_B0_renewables.csv"), index=False)
viol = b0[b0.e_max_nocurt > 1e-6]
print(f"Intervals violated without curtailment: {len(viol)} ({viol.t_h.min():.2f}-{viol.t_h.max()+dt:.2f} h); "
      f"binding: {viol.bind_nocurt.value_counts().to_dict()}")
print(f"B0 curtailment {E(b0.P_curt_kW):.1f} kWh = {E(b0.P_curt_kW)/E(day.P_res)*100:.2f} % of available RES; "
      f"peak {b0.P_curt_kW.max():.1f} kW; residual violations after curtailment: {(b0.e_max > 1e-6).sum()}")

Without storage the renewable injection is limited by the **I01-H01 Vdc-forming converter
rating** at midday: the DC-side surplus that the droop VSCs do not redistribute must leave the
DC grid through this converter, so the network-security layer curtails PV/wind there. This is
the network-forced curtailment that storage can avoid.

## Figure 2 -- nominal operating profiles

In [ ]:
FIG_WIDTH, FIG_HEIGHT = plotstyle.IEEE_COL_WIDTH_IN, 5.0
fig, axs = plt.subplots(3, 1, figsize=(FIG_WIDTH, FIG_HEIGHT), sharex=True,
                        gridspec_kw={"height_ratios": [1.0, 1.0, 0.6], "hspace": 0.12})
t = day.t0
ax = axs[0]
ax.step(t, day.P_ac, where="post", color=plotstyle.CAT["blue"], label="AC demand")
ax.step(t, day.P_dc, where="post", color=plotstyle.CAT["orange"], label="DC demand")
ax.step(t, day.P_load, where="post", color=plotstyle.INK_PRIMARY, lw=1.0, label="Total demand")
ax.set_ylabel("Demand [kW]")
ax.legend(frameon=False, fontsize=6.5, ncol=3, loc="upper left", bbox_to_anchor=(0, 1.18))
ax = axs[1]
ax.step(t, day.P_pv, where="post", color=plotstyle.CAT["yellow"], label="PV (available)")
ax.step(t, day.P_wt, where="post", color=plotstyle.CAT["green"], label="Wind (available)")
ax.fill_between(t, 0, b0.P_curt_kW, step="post", color=plotstyle.STATUS["critical"], alpha=0.35, lw=0,
                label="B0 network-forced curtailment")
ax.set_ylabel("Generation [kW]")
ax.legend(frameon=False, fontsize=6.5, loc="upper left")
ax = axs[2]
ax.step(t, day.price * 100, where="post", color=plotstyle.INK_SECONDARY)
ax.set_ylabel("Price [c/kWh]"); ax.set_ylim(0, 25)
ax.set_xlabel("Time [h]"); ax.set_xlim(0, 24); ax.set_xticks(range(0, 25, 4))
for a, lab in zip(axs, "abc"):
    a.text(-0.30, 1.02, f"({lab})", transform=a.transAxes, fontsize=8, va="top")
plotstyle.savefig(fig, os.path.join(FIG_DIR, "Fig_operating_profiles.pdf"))
plt.show()

## Takeaway

The common nominal trajectory combines the validated Case-3 demand (about 40 % of it on the DC
side) with the new DC-side PV and wind. Without storage the network can host the renewables
except around midday, when the Vdc-forming I01-H01 converter reaches its rating and part of the
renewable output must be curtailed. Every strategy is evaluated on exactly this trajectory.